# EIC Data Analysis Example

This notebook demonstrates how to analyze EIC simulation data from the public XRootD server using uproot and standard Python analysis tools.

## Data Access

EIC simulation datasets are available on the public XRootD server at `dtn-eic.jlab.org`. The data is organized by production releases in calendar versioning format (YY.MM.patch).

In [ ]:
# Import required packages
import uproot
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Set up matplotlib for better plots
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

## Listing Available Data

To see what data is available, you can use the `xrdfs` command:

```bash
# List production releases
xrdfs dtn-eic.jlab.org ls /volatile/eic/EPIC/RECO/

# List files in a specific release (e.g., 25.04.1)
xrdfs dtn-eic.jlab.org ls /volatile/eic/EPIC/RECO/25.04.1
```

In [ ]:
# Example: Access a sample DIS (Deep Inelastic Scattering) file
# This URL points to a sample file from the 25.04.1 production
base_url = "root://dtn-eic.jlab.org"
file_path = "/volatile/eic/EPIC/RECO/25.04.1/epic_craterlake/DIS/18x275/minQ2=1000/pythia8NCDIS_18x275_minQ2=1000_beamEffects_xAngle=-0.025_hiDiv_1.edm4hep.root"
file_url = f"{base_url}{file_path}"

print(f"Accessing file: {file_url}")

## Opening and Exploring the Data

EIC data files use the EDM4hep format, which is based on ROOT trees. We can use uproot to efficiently read these files.

In [ ]:
# Open the file and examine its structure
try:
    with uproot.open(file_url) as file:
        print("Available trees:")
        for key in file.keys():
            print(f"  - {key}")
        
        # Access the events tree
        if "events" in file:
            tree = file["events"]
            print(f"\nNumber of events: {tree.num_entries}")
            print("\nBranches (first 20):")
            for i, branch in enumerate(tree.keys()):
                if i < 20:
                    print(f"  - {branch}")
                elif i == 20:
                    print(f"  ... and {len(tree.keys()) - 20} more branches")
                    break
                    
except Exception as e:
    print(f"Could not access remote file: {e}")
    print("Note: This example requires network access to dtn-eic.jlab.org")
    print("For local development, you can download a sample file or create mock data")

## Sample Analysis: Reconstructed Track Analysis

Let's perform a basic analysis of reconstructed tracks in the EIC detector.

In [ ]:
# Function to analyze tracks (works with real data when available)
def analyze_tracks(file_url, max_events=1000):
    """Analyze reconstructed tracks from EIC data."""
    try:
        with uproot.open(file_url) as file:
            tree = file["events"]
            
            # Read track-related branches (adjust names based on actual data structure)
            # These are example branch names - actual names may vary
            branches_to_read = [
                "ReconstructedChargedParticles.momentum.x",
                "ReconstructedChargedParticles.momentum.y", 
                "ReconstructedChargedParticles.momentum.z",
                "ReconstructedChargedParticles.charge"
            ]
            
            # Check which branches actually exist
            available_branches = [b for b in branches_to_read if b in tree.keys()]
            
            if available_branches:
                # Read data
                arrays = tree.arrays(available_branches, entry_stop=max_events)
                return arrays
            else:
                print("Expected track branches not found. Available branches:")
                for branch in sorted(tree.keys()):
                    if "track" in branch.lower() or "particle" in branch.lower():
                        print(f"  - {branch}")
                return None
                
    except Exception as e:
        print(f"Error reading data: {e}")
        return None

# Try to analyze the data
track_data = analyze_tracks(file_url)

## Creating Sample Analysis Plots

When data is available, we can create physics plots. Here's an example with mock data to show the analysis pattern:

In [ ]:
# Create sample analysis plots (using mock data for demonstration)
np.random.seed(42)

# Generate mock track data for demonstration
n_events = 1000
n_tracks_per_event = np.random.poisson(3, n_events)  # Average 3 tracks per event

# Generate physics-realistic distributions
px = np.random.normal(0, 1.5, np.sum(n_tracks_per_event))  # GeV/c
py = np.random.normal(0, 1.5, np.sum(n_tracks_per_event))  # GeV/c
pz = np.random.exponential(2.0, np.sum(n_tracks_per_event))  # Forward-boosted
charge = np.random.choice([-1, 1], np.sum(n_tracks_per_event))

# Calculate derived quantities
pt = np.sqrt(px**2 + py**2)  # Transverse momentum
p_total = np.sqrt(px**2 + py**2 + pz**2)  # Total momentum
eta = np.arcsinh(pz / pt)  # Pseudorapidity
phi = np.arctan2(py, px)  # Azimuthal angle

print(f"Generated {len(px)} tracks from {n_events} events")
print(f"Average tracks per event: {len(px)/n_events:.2f}")

In [ ]:
# Create physics analysis plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Transverse momentum distribution
axes[0,0].hist(pt, bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_xlabel('Transverse Momentum $p_T$ (GeV/c)')
axes[0,0].set_ylabel('Number of Tracks')
axes[0,0].set_title('Track $p_T$ Distribution')
axes[0,0].grid(True, alpha=0.3)

# Pseudorapidity distribution
axes[0,1].hist(eta, bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[0,1].set_xlabel('Pseudorapidity $\\eta$')
axes[0,1].set_ylabel('Number of Tracks')
axes[0,1].set_title('Track $\\eta$ Distribution')
axes[0,1].grid(True, alpha=0.3)

# Momentum vs pseudorapidity (2D)
h = axes[1,0].hist2d(eta, p_total, bins=30, cmap='Blues')
axes[1,0].set_xlabel('Pseudorapidity $\\eta$')
axes[1,0].set_ylabel('Total Momentum $p$ (GeV/c)')
axes[1,0].set_title('Momentum vs Pseudorapidity')
plt.colorbar(h[3], ax=axes[1,0], label='Number of Tracks')

# Charge distribution
charge_counts = np.bincount(charge + 1)  # +1 to handle negative indices
axes[1,1].bar(['Negative', 'Positive'], [charge_counts[0], charge_counts[2]], 
              color=['red', 'blue'], alpha=0.7, edgecolor='black')
axes[1,1].set_ylabel('Number of Tracks')
axes[1,1].set_title('Track Charge Distribution')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Advanced Analysis Example: Invariant Mass

A common physics analysis is calculating invariant masses of particle pairs.

In [ ]:
# Calculate invariant masses for opposite-charge pairs
# Assume pion mass for simplicity
pion_mass = 0.139570  # GeV/c²

# Calculate 4-momentum components
E = np.sqrt(p_total**2 + pion_mass**2)  # Energy assuming pion mass

# Find opposite charge pairs
pos_indices = np.where(charge == 1)[0]
neg_indices = np.where(charge == -1)[0]

invariant_masses = []

# Calculate invariant mass for all opposite-charge pairs
for i in pos_indices[:100]:  # Limit to first 100 for speed
    for j in neg_indices[:100]:
        if i != j:  # Don't pair particle with itself
            # Calculate invariant mass
            E_sum = E[i] + E[j]
            px_sum = px[i] + px[j]
            py_sum = py[i] + py[j]
            pz_sum = pz[i] + pz[j]
            
            inv_mass = np.sqrt(E_sum**2 - (px_sum**2 + py_sum**2 + pz_sum**2))
            invariant_masses.append(inv_mass)

invariant_masses = np.array(invariant_masses)
print(f"Calculated {len(invariant_masses)} invariant masses")

In [ ]:
# Plot invariant mass spectrum
plt.figure(figsize=(10, 6))
plt.hist(invariant_masses, bins=50, alpha=0.7, edgecolor='black', color='green')
plt.xlabel('Invariant Mass (GeV/c²)')
plt.ylabel('Number of Pairs')
plt.title('Invariant Mass Spectrum of Opposite-Charge Pairs')
plt.grid(True, alpha=0.3)

# Add some physics context
plt.axvline(x=0.547862, color='red', linestyle='--', alpha=0.7, label='η meson mass')
plt.axvline(x=0.782659, color='blue', linestyle='--', alpha=0.7, label='ρ meson mass')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Mean invariant mass: {np.mean(invariant_masses):.3f} GeV/c²")
print(f"RMS: {np.std(invariant_masses):.3f} GeV/c²")

## Data Access Summary

This notebook demonstrates the pattern for EIC data analysis:

1. **Access Data**: Use XRootD URLs to access files from `dtn-eic.jlab.org`
2. **Read Efficiently**: Use uproot for fast ROOT file reading
3. **Physics Analysis**: Calculate derived quantities (pT, η, φ, invariant masses)
4. **Visualization**: Create physics-meaningful plots

### Key Commands for Real Data Access:

```bash
# List available productions
xrdfs dtn-eic.jlab.org ls /volatile/eic/EPIC/RECO/

# List files in latest production (adjust version number)
xrdfs dtn-eic.jlab.org ls /volatile/eic/EPIC/RECO/25.04.1/epic_craterlake/
```

### Next Steps:
- Explore different event types (DIS, exclusive processes, etc.)
- Analyze calorimeter data for energy measurements
- Study particle identification algorithms
- Compare with Monte Carlo truth information
- Implement physics-specific selections and cuts